# 🚀 Dual Model Trainer (Segmentor + Classifier)

In this notebook, we implement an end-to-end Dual Model training and evaluation system:
1. **Stage 1 (Segmentor):** A YOLOv26m-seg (or any YOLO architecture) instance segmentation model trained on a generalized "1-class Rock" dataset.
2. **Stage 2 (Classifier):** Fine-grained Image Classification models trained natively on rock instance *crops*.

We benchmark 6 Classifier architectures specifically fine-tuned for small rock crops:
- `YOLOv26-cls`
- `convnext_tiny`
- `tf_efficientnetv2_s`
- `resnet50`
- `densenet121`
- `davit_tiny`

**Note:** The dataset generation steps (converting generalized 6-class YOLO dataset into single class &amp; cropping bounding boxes into class folders) are assumed to be complete. 

In [1]:
from pathlib import Path
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, vit_b_16
import copy
from ultralytics import YOLO
import gc

# ==========================================
# 📝 PATH CONFIGURATION - Adjust as necessary
# ==========================================
BASE_DIR = Path.home() / "projects" / "DwiAnggara" / "Datasets" 

# ── 1. SEGMENTOR PATHS ──
# YOLO dataset containing only 1-class mapping ("rock") 
SEGMENTOR_DATASET_YAML = BASE_DIR / "Batch3and4_YOLO_SingleClass" / "data.yaml"

# ── 2. CLASSIFIER PATHS ──
# Structured as PyTorch ImageFolder (e.g. train/Silt/..., val/Sandstone/...)
CLASSIFIER_DATA_ROOT = BASE_DIR / "Batch3and4_Classifier"

# ── TARGET MODEL NAMES ──
SEGMENTOR_RUN_NAME = "YOLO_Segmentor_SingleClass_Final"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Basic Setup Validation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   : {device}")
print(f"Base Dir : {BASE_DIR}")
print(f"CUDA     : {torch.cuda.is_available()}")

Device   : cuda
Base Dir : /home/praktikan/projects/DwiAnggara/Datasets
CUDA     : True


## 🔧 Stage 1: Segmentor Training (Single-Class YOLO)
> **Goal**: Train YOLO to detect general boundaries of "Rocks", completely ignoring fine-grained taxonomy.

In [2]:
def train_segmentor(epochs=100, imgsz=640, batch=4, workers=2):
    """
    Trains the Model 1 (Segmentor) relying on Ultralytics YOLO logic natively.
    Optimized parameters to prevent CUDA Out Of Memory (OOM).
    """
    print("\n" + "=" * 50)
    print("🚀 TRAINING STAGE 1: SEGMENTOR")
    print("=" * 50)
    
    # Optional check if dataset YAML exists before attempting
    if not SEGMENTOR_DATASET_YAML.exists():
        print(f"⚠️ Warning: Cannot initiate Stage 1 - Dataset not found: {SEGMENTOR_DATASET_YAML}")
        return None
        
    # [OPTIMIZATION] Free system RAM and CUDA VRAM before YOLO allocates memory
    torch.cuda.empty_cache()
    gc.collect()
        
    model = YOLO("yolo11m-seg.pt") # fallback/starter
    
    results = model.train(
        data=str(SEGMENTOR_DATASET_YAML),
        project=str(MODELS_DIR),
        name=SEGMENTOR_RUN_NAME,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        workers=workers,
        device=0 if device == 'cuda' else 'cpu',
        single_cls=True,  # Critical: Forces YOLO to treat all existing ground-truths as class 0
        exist_ok=True,
        verbose=False
    )
    
    # [OPTIMIZATION] Explicitly delete model from memory and clear cache
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return results

seg_results = train_segmentor(epochs=100)


🚀 TRAINING STAGE 1: SEGMENTOR
New https://pypi.org/project/ultralytics/8.4.37 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.35 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO_SingleClass/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0

## 🧠 Stage 2: Classifier Training Data Loader
> **Goal**: Preload and transform our cropped rock patches for PyTorch models

In [3]:
# PyTorch ImageFolder structure requires Train & Val loaders
batch_size = 32

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

def load_classifier_datasets():
    if not CLASSIFIER_DATA_ROOT.exists():
        print(f"⚠️ Warning: Crop directory {CLASSIFIER_DATA_ROOT} missing.")
        return None, None, None

    image_datasets = {
        x: datasets.ImageFolder(os.path.join(CLASSIFIER_DATA_ROOT, x), data_transforms[x]) 
        for x in ['train', 'val']
    }
    
    # [OPTIMIZATION] Reduced num_workers from 4 to 2 to minimize System RAM usage 
    dataloaders = {
        x: DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=2) 
        for x in ['train', 'val']
    }
    
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
    class_names = image_datasets['train'].classes
    return dataloaders, dataset_sizes, class_names

dataloaders, dataset_sizes, class_names = load_classifier_datasets()

## 🚀 Stage 2: Train Expert Classifiers

Here we build a standard reusable PyTorch loop and use Ultralytics for the YOLO-cls version. 

### A) PyTorch Reusable Training Loop (EfficientNet &amp; ViT)

### B) Classifier 1: Ultralytics YOLO-cls

In [4]:
def train_yolo_classifier(epochs=50, imgsz=224, batch=16, workers=2):
    print("\n" + "=" * 50)
    print("🚀 TRAINING STAGE 2: YOLO-cls")
    print("=" * 50)
    
    if not CLASSIFIER_DATA_ROOT.exists(): return None
        
    torch.cuda.empty_cache()
    gc.collect()
        
    model = YOLO('yolo26n-cls.pt')
    results = model.train(
        data=str(CLASSIFIER_DATA_ROOT),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        workers=workers,
        project=str(MODELS_DIR),
        name="Expert_YOLOcls_Final"
    )
    
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return results

cls_yolo_res = train_yolo_classifier(epochs=25)


🚀 TRAINING STAGE 2: YOLO-cls
New https://pypi.org/project/ultralytics/8.4.37 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.35 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_Classifier, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mo

### Comparative Evaluation (MultiModel architectures vs YOLO-cls)
We can compare the validation top-1 accuracy / F1 metrics for the models trained.

### C) Classifier 2 & 3: PyTorch Models via MultiModelTrainer

In [5]:
import sys
# Ensure src is in the path
sys.path.append(str(Path(os.path.abspath('')).parent.parent))
from src.MultiModelImageClassification import MultiModelTrainer

def train_multimodel(model_name, dataloaders, epochs=20):
    print("\n" + "=" * 50)
    print(f"🚀 TRAINING STAGE 2: {model_name}")
    print("=" * 50)
    
    num_classes = len(dataloaders['train'].dataset.classes)
    trainer = MultiModelTrainer(
        model_name=model_name,
        num_classes=num_classes,
        device=device,
        pretrained=True
    )
    
    # Train
    history = trainer.train(
        dataloaders['train'], dataloaders['val'],
        epochs=epochs,
        lr=1e-4,
        loss_fn='cross_entropy'
    )
    
    # Save the best weights
    model_save_path = MODELS_DIR / f"Expert_{model_name}_Final.pth"
    torch.save(trainer.model.state_dict(), str(model_save_path))
    print(f"Saved {model_name} to {model_save_path}")
    
    return trainer.model, history

# Using supported models from the factory: 'tf_efficientnetv2_s', 'davit_tiny', etc.
# Instead of efficientnet_b0 and vit_b_16 (which were custom defined), we use the models supported by the class:
efficientnet_model, eff_hist = train_multimodel('tf_efficientnetv2_s', dataloaders, epochs=25)
vit_model, vit_hist = train_multimodel('davit_tiny', dataloaders, epochs=25)
convnext_model, convnext_hist = train_multimodel('convnext', dataloaders, epochs=25)



🚀 TRAINING STAGE 2: tf_efficientnetv2_s

Epoch 1/25
------------------------------


Validating tf_efficientnetv2_s: 100%|██████████| 39/39 [00:01<00:00, 21.54it/s, Loss=0.7626]


Train Loss: 0.6846, Acc: 0.7386, F1: 0.7371
Val Loss: 0.7986, Acc: 0.6889, F1: 0.6779
Saved best model with Val F1: 0.6779

Epoch 2/25
------------------------------


Validating tf_efficientnetv2_s: 100%|██████████| 39/39 [00:01<00:00, 22.61it/s, Loss=0.6823]


Train Loss: 0.4373, Acc: 0.8323, F1: 0.8318
Val Loss: 0.7237, Acc: 0.7419, F1: 0.7339
Saved best model with Val F1: 0.7339

Epoch 3/25
------------------------------


...

Validating davit_tiny: 100%|██████████| 39/39 [00:01<00:00, 22.09it/s, Loss=0.5279]


Train Loss: 0.0008, Acc: 0.9998, F1: 0.9998
Val Loss: 1.4200, Acc: 0.7687, F1: 0.7649

Epoch 25/25
------------------------------


Validating davit_tiny: 100%|██████████| 39/39 [00:01<00:00, 20.80it/s, Loss=2.0649]


Train Loss: 0.0008, Acc: 0.9998, F1: 0.9998
Val Loss: 1.4190, Acc: 0.7687, F1: 0.7649
Saved davit_tiny to /home/praktikan/projects/DwiAnggara/Datasets/models/Expert_davit_tiny_Final.pth

🚀 TRAINING STAGE 2: convnext


RuntimeError: Unknown model (convnext)

In [6]:
import pandas as pd

def show_comparison(yolo_results, eff_hist, vit_hist):
    print("\n" + "=" * 50)
    print("📊 COMPARATIVE EVALUATION SUMMARY")
    print("=" * 50)
    
    # We retrieve the best metrics
    # YOLO results is a Results object containing metrics.top1
    yolo_acc = getattr(yolo_results, "top1", 0.0) if yolo_results else 0.0
    
    eff_best_acc = max(eff_hist['val_acc']) if eff_hist else 0.0
    eff_best_f1  = max(eff_hist['val_f1']) if eff_hist else 0.0
    
    vit_best_acc = max(vit_hist['val_acc']) if vit_hist else 0.0
    vit_best_f1  = max(vit_hist['val_f1']) if vit_hist else 0.0
    
    data = {
        "Model Architecture": ["YOLOv26-cls", "tf_efficientnetv2_s", "davit_tiny"],
        "Best Val Accuracy": [f"{yolo_acc:.4f}", f"{eff_best_acc:.4f}", f"{vit_best_acc:.4f}"],
        "Best Val F1-Score": ["N/A", f"{eff_best_f1:.4f}", f"{vit_best_f1:.4f}"]
    }
    
    df = pd.DataFrame(data)
    display(df)

# Run the comparison script
show_comparison(cls_yolo_res, eff_hist, vit_hist)



📊 COMPARATIVE EVALUATION SUMMARY


,Model Architecture,Best Val Accuracy,Best Val F1-Score
0,YOLOv26-cls,0.7508,N/A
1,tf_efficientnetv2_s,0.7549,0.7513
2,davit_tiny,0.7728,0.7691


## 📊 End-to-End Test Evaluation
We evaluate the complete system (Segmentor + Classifier) metrics: **Precision, Recall, F1-Score, and mAP**.

Since doing this directly requires matching predicted segment masks tightly with test set labels using IoU logic on our custom classes, below is a mock script detailing how `sklearn.metrics` will calculate this out of our Integration pipeline predictions.

In [7]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

def mock_evaluation_report(y_true, y_pred, class_names):
    """
    Computes strict metrics for validation.
    y_true: list of integer target labels for crops evaluated
    y_pred: list of integer predictions mapped by the classifier
    """
    p = precision_score(y_true, y_pred, average='weighted')
    r = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print("="*40)
    print("      INTEGRATION APP EVALUATION      ")
    print("="*40)
    print(f"mAP (Approximated):  ~{f1*100:.2f}%")
    print(f"Precision:           {p*100:.2f}%")
    print(f"Recall:              {r*100:.2f}%")
    print(f"F1-Score:            {f1*100:.2f}%")
    print("="*40)
    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))


## 🎯 Integration Pipeline & Visualizations
We combine the Segmentor and Classifiers to identify boundaries and classify crops!

In [8]:
import sys
import gc
import cv2
import random
from pathlib import Path
from IPython.display import display, Image

# Setup Project Path
sys.path.append(str(BASE_DIR))
from src.inference_dual_model import DualModelPipeline, DualModelVisualizer

# Define Model Loading Paths
SEG_MODEL_PATH = str(MODELS_DIR / SEGMENTOR_RUN_NAME / "weights" / "best.pt")
YOLO_CLS_PATH  = str(MODELS_DIR / "Expert_YOLOcls_Final" / "weights" / "best.pt")

# Master Schema Map (6 Classes format)
class_map = {
    0: 'CLAS - Silt',
    1: 'CLAS - Sandstone', 
    2: 'CARB - Limestone',
    3: 'ORG - Coal',
    4: 'CLAS - Shalestone',
    5: 'MIN - Quartz',
}

configs = [
    {
        'name': 'YOLO-cls', 
        'model': YOLO_CLS_PATH, 
        'type': 'yolo', 
        'class_names': class_map
    },
    {
        'name': 'EfficientNetV2', 
        'model': str(MODELS_DIR / "Expert_tf_efficientnetv2_s_Final.pth"), 
        'type': 'pytorch_timm', 
        'class_names': class_map, 
        'architecture': 'tf_efficientnetv2_s'
    },
    {
        'name': 'DaViT', 
        'model': str(MODELS_DIR / "Expert_davit_tiny_Final.pth"), 
        'type': 'pytorch_timm', 
        'class_names': class_map, 
        'architecture': 'davit_tiny'
    }
]

# Create Output Predictions Directory
OUTPUT_PRED_DIR = BASE_DIR / "predictions_dual_model"
OUTPUT_PRED_DIR.mkdir(parents=True, exist_ok=True)

# Ensure Segmentor exists before pipeline initialization
if not os.path.exists(SEG_MODEL_PATH):
    print(f"⚠️ Segmentor weight not found at {SEG_MODEL_PATH}. Cannot build Pipeline.")
    pipeline = None
else:
    # Initialize the reusable inference pipeline globally so it doesn't load/unload aggressively
    pipeline = DualModelPipeline(segmentor_path=SEG_MODEL_PATH, classifier_configs=configs)
    
# Initialize bounding box visualizer
visualizer = DualModelVisualizer(color_map=None)

Loading Segmentor: /home/praktikan/projects/DwiAnggara/Datasets/models/YOLO_Segmentor_SingleClass_Final/weights/best.pt
Loading Classifier [YOLO-cls] (Type: yolo)
Loading Classifier [EfficientNetV2] (Type: pytorch_timm)
Loading Classifier [DaViT] (Type: pytorch_timm)


In [10]:
import io
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# UI Components for Interactive Inference
upload_btn = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)

classifier_dropdown = widgets.Dropdown(
    options=['YOLO-cls', 'EfficientNetV2', 'DaViT'],
    value='EfficientNetV2',
    description='Classifier:',
    disabled=False,
)

run_button = widgets.Button(
    description='Run Inference',
    button_style='success',
    icon='play'
)

output_plot = widgets.Output()

def run_interactive_visualization(b):
    with output_plot:
        clear_output(wait=True)
        
        if pipeline is None:
            print("⚠️ Initialization failed. The pipeline is empty. Check YOLO Segmentor pipeline cell above.")
            return

        if not upload_btn.value:
            print("⚠️ Please upload an image first.")
            return
            
        print(f"⏳ Processing via {classifier_dropdown.value} architecture...")
        
        # 1. Parse Uploaded Image
        # Handle difference between ipywidgets versions (dict vs tuple)
        uploaded_file = list(upload_btn.value.values())[0] if isinstance(upload_btn.value, dict) else upload_btn.value[0]
        content = uploaded_file['content']
        file_name = uploaded_file['name']
        
        # Decode byte array to cv2 image
        nparr = np.frombuffer(content, np.uint8)
        original_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        # Optional Explicit Cache Freeing before running logic
        torch.cuda.empty_cache()
        gc.collect()

        # Step 2: Save to a temporary path (to support pipeline's file path requirements)
        temp_path = str(OUTPUT_PRED_DIR / f"temp_{file_name}")
        cv2.imwrite(temp_path, original_bgr)

        try:
            # Step 3: Run Pipeline End-to-End
            results, predictions = pipeline.predict(image=temp_path, classifier_name=classifier_dropdown.value)
            
            # Step 4: Overlay Post-Processed Display Masks
            drawn_img = visualizer.draw(image=original_bgr, predictions=predictions, thickness=2)
            
            # Step 5: Export the Visualization Artifact Format
            exported_img_name = f"pred_combo_{classifier_dropdown.value}_{file_name}"
            final_output_path = OUTPUT_PRED_DIR / exported_img_name
            cv2.imwrite(str(final_output_path), drawn_img)
            
            print(f"✅ Prediction completed! Image exported to -> {final_output_path}")
            
            # Standard output Display Image within notebook 
            display(Image(filename=str(final_output_path)))
            
        except Exception as e:
            print(f"❌ An error occurred during inference: {e}")
        finally:
            # Cleanup temp file
            if os.path.exists(temp_path):
                os.remove(temp_path)

# Attach click event
run_button.on_click(run_interactive_visualization)

# Display UI Structure
ui = widgets.VBox([
    widgets.HTML("<b>Upload Raw Rock Image for Full Pipeline Inference</b>"),
    widgets.HTML("<p>Upload a whole image, and it will be masked by the YOLO segmentor, cropped automatically, and classified on the fly.</p>"),
    upload_btn,
    classifier_dropdown,
    widgets.HTML("<br>"),
    run_button,
    output_plot
])

display(ui)